In [7]:
import os
import fnmatch

def delete_non_filtered_tif_files(folder_path):
    # Deletes all files starting with 'Non-Filtered' and with the '.tif' extension
    # from the specified folder and its subfolders.
    if not os.path.isdir(folder_path):
        print(f"Error: The folder '{folder_path}' does not exist.")
        return

    # Traverse the folder and its subfolders
    for root, dirs, files in os.walk(folder_path):
        for filename in files:
            # Check if the file matches the naming criteria
            if fnmatch.fnmatch(filename, 'Non-Filtered*.tif'):
                file_path = os.path.join(root, filename)
                try:
                    os.remove(file_path)
                    print(f"Deleted: {file_path}")
                except Exception as e:
                    print(f"Error deleting file '{file_path}': {e}")

if __name__ == "__main__":
    
    target_folder = '/Users/chenruizhen/project_data' 
    delete_non_filtered_tif_files(target_folder)


Error: The folder '/Users/chenruizhen/project_data' does not exist.


In [14]:
import os
import random
import shutil

# Define directory paths
random_series_path = './project_data/Random_series'
flux_series_path = './project_data/Flux_series'
time_series_path = './project_data/Time_series'
two_phase_series_path = './project_data/Two_phase_series'
boundary_series_path = './project_data/Boundary_series'

val_output_path = './project_data/validation_set'
test_output_path = './project_data/test_set'
train_output_path = './project_data/train_set'

# Define the split ratios
val_ratio = 1
test_ratio = 2
train_ratio = 7

# Create output directories if they do not exist
os.makedirs(val_output_path, exist_ok=True)
os.makedirs(test_output_path, exist_ok=True)
os.makedirs(train_output_path, exist_ok=True)

# Calculate total ratio
total_ratio = val_ratio + test_ratio + train_ratio

# Get list of second-level subfolders from Random_series
random_series_folders = [f.path for f in os.scandir(random_series_path) if f.is_dir()]

# Shuffle the list of subfolders randomly
random.shuffle(random_series_folders)

# Calculate the number of folders for each dataset
total_folders = len(random_series_folders)
val_count = int((val_ratio / total_ratio) * total_folders)
test_count = int((test_ratio / total_ratio) * total_folders)
train_count = total_folders - val_count - test_count  # The rest goes to training set

# Split the Random_series dataset into validation, test, and training sets
val_folders = random_series_folders[:val_count]
test_folders = random_series_folders[val_count:val_count + test_count]
train_folders = random_series_folders[val_count + test_count:]

# Function to get second-level subfolders from other series directories
def get_subfolders(series_path):
    return [f.path for f in os.scandir(series_path) if f.is_dir()]

# Add subfolders from Flux_series, Time_series, Two_phase_series, and Boundary_series to the training set
train_folders += get_subfolders(flux_series_path)
train_folders += get_subfolders(time_series_path)
train_folders += get_subfolders(two_phase_series_path)
train_folders += get_subfolders(boundary_series_path)

# Function to copy folders to the target output directory
def copy_folders(folders, output_path):
    for folder in folders:
        folder_name = os.path.basename(folder)
        dest_path = os.path.join(output_path, folder_name)
        # Check if the folder already exists, to avoid duplication
        if not os.path.exists(dest_path):
            shutil.copytree(folder, dest_path)
        else:
            print(f"Folder already exists, skipping: {dest_path}")

# Copy validation, test, and training folders to their respective directories
copy_folders(val_folders, val_output_path)
copy_folders(test_folders, test_output_path)
copy_folders(train_folders, train_output_path)

# Print the count of folders in each dataset
print(f"Validation set count: {len(val_folders)}, Test set count: {len(test_folders)}, Training set count: {len(train_folders)}")


验证集数量: 20, 测试集数量: 40, 训练集数量: 238
